# DMV Practical 6 – Retail Sales Analysis

**Aim:** Select and rename relevant retail transaction columns, clean data, aggregate sales by region and product category, and visualize sales performance.

The PDF shows a retail transaction dataset with columns such as CustomerRegion, ProductCategory and FinalAmount. If the original URL does not load, upload the same retail sales CSV; the notebook maps those fields automatically.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

url = "https://raw.githubusercontent.com/simranhaldar/retail-sales-analysis/main/data/retail_sales.csv"
try:
    df = pd.read_csv(url)
except Exception:
    print("Upload the retail sales CSV used in the practical.")
    from google.colab import files
    uploaded = files.upload()
    df = pd.read_csv(next(iter(uploaded)))

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
df.info()
display(df.head())

In [ ]:
# Inspect quality
print("Missing values:\n", df.isnull().sum())
print("Duplicate rows:", df.duplicated().sum())
print("Relevant variables for aggregation:")
for col in ["Region", "Sales", "Category", "Order Date"]:
    if col in df.columns:
        print(col)

# Map available source fields to the practical's required fields
aliases = {
    "CustomerRegion": "Region",
    "ProductCategory": "Category",
    "FinalAmount": "Sales",
    "TransactionDate": "Order Date",
    "Quantity": "Quantity Sold"
}
sales_df = df.rename(columns={k:v for k,v in aliases.items() if k in df.columns}).copy()
print("Actual columns:", sales_df.columns.tolist())
display(sales_df.head())

In [ ]:
# Convert types and clean
if "Sales" in sales_df.columns:
    sales_df["Sales"] = pd.to_numeric(sales_df["Sales"], errors="coerce")
if "Quantity Sold" in sales_df.columns:
    sales_df["Quantity Sold"] = pd.to_numeric(sales_df["Quantity Sold"], errors="coerce")
if "Order Date" in sales_df.columns:
    sales_df["Order Date"] = pd.to_datetime(sales_df["Order Date"], errors="coerce", dayfirst=True)

needed = [c for c in ["Region","Category","Sales","Order Date","Quantity Sold"] if c in sales_df.columns]
sales_df = sales_df.dropna(subset=[c for c in ["Region","Category","Sales"] if c in sales_df.columns])
print("Shape after cleaning:", sales_df.shape)
display(sales_df.head())

In [ ]:
# Sales by region
if {"Region","Sales"}.issubset(sales_df.columns):
    region_sales = sales_df.groupby("Region")["Sales"].sum().sort_values(ascending=False)
    display(region_sales)
    region_sales.plot(kind="bar", figsize=(6,3), title="Total Sales by Region")
    plt.xlabel("Region"); plt.ylabel("Total Sales"); plt.xticks(rotation=30)
    plt.tight_layout(); plt.show()

    region_sales.plot(kind="pie", autopct="%1.1f%%", figsize=(5,5), title="Sales Distribution by Region")
    plt.ylabel(""); plt.show()

In [ ]:
# Top regions and sales by region/category
if {"Region","Sales"}.issubset(sales_df.columns):
    top_regions = region_sales.head(3)
    display(top_regions)
    if len(top_regions):
        print("Top performing region:", top_regions.index[0])
        print("Total sales:", round(top_regions.iloc[0], 2))

if {"Region","Category","Sales"}.issubset(sales_df.columns):
    region_category_sales = sales_df.groupby(["Region","Category"])["Sales"].sum().reset_index()
    display(region_category_sales)
    pivot_sales = region_category_sales.pivot(index="Region", columns="Category", values="Sales").fillna(0)
    display(pivot_sales)
    pivot_sales.plot(kind="bar", stacked=True, figsize=(8,4), title="Sales by Region and Product Category")
    plt.ylabel("Total Sales"); plt.xticks(rotation=30); plt.tight_layout(); plt.show()
    pivot_sales.plot(kind="bar", stacked=False, figsize=(8,4), title="Comparison of Sales by Region and Product Category")
    plt.ylabel("Total Sales"); plt.xticks(rotation=30); plt.tight_layout(); plt.show()

In [ ]:
# Total sales by category and final summary
if {"Category","Sales"}.issubset(sales_df.columns):
    category_sales = sales_df.groupby("Category")["Sales"].sum().sort_values(ascending=False)
    display(category_sales)
    category_sales.plot(kind="bar", figsize=(7,3), title="Total Sales by Product Category")
    plt.xlabel("Product Category"); plt.ylabel("Total Sales"); plt.xticks(rotation=30)
    plt.tight_layout(); plt.show()

if "Sales" in sales_df.columns:
    print("Total Sales:", round(sales_df["Sales"].sum(), 2))
    print("Average Sale:", round(sales_df["Sales"].mean(), 2))

In [ ]:
# ========== DMV PRACTICAL 6: RETAIL SALES ANALYSIS ==========

# AIM: Clean retail data, analyze sales by region/category,
# identify top regions and visualize sales performance.
# FLOW: Load -> Inspect -> Rename -> Clean -> Group -> Analyze -> Plot

# THEORY:
# Data Wrangling = cleaning and organizing raw data.
# Data Aggregation = combining values to calculate totals/averages.
# Grouping = organizing similar records together.
# Pivot Table = summarizes data in rows and columns.
# Sales Analysis = studying sales to identify business performance.

# LIBRARIES:
# pandas(pd) = data handling; numpy(np) = numerical operations.
# matplotlib(plt) = graphs and visualization.
# NOTE: np is imported but not directly used.

# DATA LOADING:
# try/except = handles errors while loading CSV.
# pd.read_csv(url) = loads CSV into DataFrame.
# files.upload() = manually uploads CSV in Google Colab.
# next(iter(uploaded)) = gets first uploaded filename.
# shape = (rows,columns); columns.tolist() = column names as list.
# info() = data types and non-null counts.
# head() = first 5 rows; display() = shows formatted table.
# isnull().sum() = missing values per column.
# duplicated().sum() = counts duplicate rows.
# NOTE: Code checks duplicates but does NOT remove them.

# COLUMN RENAMING:
# aliases = dictionary mapping old names to new names.
# rename(columns={k:v ...}) = renames existing columns.
# k = original column name; v = new column name.
# if k in df.columns = checks whether column exists.
# CustomerRegion -> Region; ProductCategory -> Category.
# FinalAmount -> Sales; TransactionDate -> Order Date.
# Quantity -> Quantity Sold.
# copy() = creates an independent DataFrame copy.

# DATA CLEANING:
# pd.to_numeric(...,errors="coerce") = converts to numbers.
# pd.to_datetime(...,errors="coerce",dayfirst=True) = converts dates.
# errors="coerce" = invalid values become NaN/NaT.
# dayfirst=True = interprets dates as day-month-year.
# dropna(subset=[...]) = removes rows missing required values.
# Required columns = Region, Category and Sales (if present).
# needed = list of existing relevant columns (not used later).
# issubset(sales_df.columns) = checks required columns exist.

# REGION-WISE SALES:
# groupby("Region")["Sales"].sum() = total sales per region.
# sort_values(ascending=False) = highest to lowest sales.
# plot(kind="bar") = vertical bar chart.
# plot(kind="pie") = pie chart showing sales proportions.
# autopct="%1.1f%%" = percentage with 1 decimal place.
# figsize=(6,3) = graph size in inches.
# xlabel()/ylabel() = axis labels.
# xticks(rotation=30) = rotates X-axis labels by 30 degrees.
# title = chart heading; tight_layout() = adjusts spacing.
# show() = displays graph.

# TOP PERFORMING REGIONS:
# region_sales.head(3) = top 3 highest-selling regions.
# len(top_regions) = number of regions returned.
# top_regions.index[0] = name of highest-selling region.
# top_regions.iloc[0] = sales value of highest region.
# round(value,2) = rounds to 2 decimal places.

# REGION & CATEGORY ANALYSIS:
# groupby(["Region","Category"])["Sales"].sum()
# = calculates total sales for each region-category combination.
# reset_index() = converts grouped index into normal columns.
# pivot(index="Region",columns="Category",values="Sales")
# = creates a table with regions as rows and categories as columns.
# fillna(0) = replaces missing combinations with zero.
# stacked=True = category bars stacked on each other.
# stacked=False = category bars displayed side by side.
# Bar chart = compares sales across regions/categories.

# CATEGORY-WISE SALES:
# groupby("Category")["Sales"].sum() = total sales per category.
# sort_values(ascending=False) = sorts highest sales first.
# plot(kind="bar") = displays category sales comparison.

# FINAL SUMMARY:
# sales_df["Sales"].sum() = total revenue/sales.
# sales_df["Sales"].mean() = average sales per transaction.
# round(...,2) = displays 2 decimal places.

# ========== 4 IMPORTANT VIVA QUESTIONS ==========

# Q1. What is Data Aggregation?
# Ans: Combining multiple records to calculate totals,
#      averages, counts or other summary values.

# Q2. What is groupby() in Pandas?
# Ans: It groups similar records and applies operations
#      like sum(), mean() or count().

# Q3. What is the difference between Bar Chart and Pie Chart?
# Ans: Bar Chart compares values across categories.
#      Pie Chart shows each category's share of the total.

# Q4. What is a Pivot Table?
# Ans: A table that summarizes data using rows and columns.
#      Here, rows = Regions, columns = Categories,
#      values = total Sales.

# FINAL VIVA EXPLANATION:
# "We load retail sales data, inspect missing values and
# duplicates, rename columns, convert data types and remove
# rows with missing required values. Then we calculate
# region-wise and category-wise sales using groupby(),
# identify top-performing regions, create pivot tables,
# visualize results using bar and pie charts, and calculate
# total and average sales."